# 05 — Backtest walk-forward có phí giao dịch

Notebook chạy ba chiến lược long-only với cửa sổ 24 tháng, lag quyết định một tháng, tái cân bằng hàng tháng và bốn mức phí. VN30 là benchmark chính. Runner kiểm tra cấu hình, không cho ghi đè thư mục kết quả và lưu hash đầu vào.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from prepare_vn100_backtest_inputs import prepare
from run_backtest import execute

## Chạy có kiểm soát

Input và output dành riêng cho notebook. Nếu output đã tồn tại, notebook chỉ đọc lại; muốn chạy mới, hãy đổi tên output trong cấu hình hoặc di chuyển thư mục cũ.

In [ ]:
INPUT = ROOT / 'local_runs' / 'notebook_vn100_inputs'
OUTPUT = ROOT / 'local_runs' / 'notebook_vn100_backtest'
CONFIG = ROOT / 'local_runs' / 'notebook_backtest_config.json'
if not INPUT.exists():
    prepare(ROOT / 'outputs' / 'vn_period_factors', INPUT)

config = {
    'synthetic': False,
    'universe_mode': 'dynamic',
    'data_source': 'Versioned VN100 period-factor outputs from supplied Refinitiv LSEG archives',
    'benchmark_name': 'VN30 monthly price return',
    'return_basis': 'Close-EOM monthly simple return; reported fallback where price return is unavailable',
    'rf_basis': 'Effective monthly return converted from supplied annual 1Y government yield',
    'end': '2026-06-30',
    'universe_selection': 'VN100 membership observed for each deployment month',
    'execution_assumption': 'Weights formed with a one-month decision lag and applied at month end',
    'returns': 'notebook_vn100_inputs/returns.csv',
    'membership': 'notebook_vn100_inputs/membership.csv',
    'benchmark': 'notebook_vn100_inputs/benchmark_vn30.csv',
    'risk_free': 'notebook_vn100_inputs/risk_free.csv',
    'output_dir': 'notebook_vn100_backtest',
    'lookback': 24,
    'rebalance_every': 1,
    'decision_lag_periods': 1,
    'rates': [0, 0.0015, 0.0025, 0.0035],
    'strategies': {
        'MinVariance': 'builtin:min_variance',
        'MaxSharpe': 'builtin:max_sharpe',
        'EqualWeight': 'builtin:equal_weight',
    },
    'ridge': 1e-8,
}
if not OUTPUT.exists():
    CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    execute(CONFIG)
OUTPUT

In [ ]:
comparison = pd.read_csv(OUTPUT / 'comparison.csv')
strategy_rows = comparison.loc[(comparison['name'] == 'strategy') & (comparison['transaction_cost_rate'] == 0.0025)].copy()
benchmark_row = comparison.loc[comparison['name'] == 'benchmark'].head(1).copy()
benchmark_row['strategy'] = 'VN30'
main_fee = pd.concat([strategy_rows, benchmark_row], ignore_index=True)
display(main_fee[['strategy', 'name', 'transaction_cost_rate', 'months', 'sharpe', 'cagr', 'max_drawdown', 'total_return']])
display(comparison.loc[comparison['name'] == 'strategy'].pivot(index='transaction_cost_rate', columns='strategy', values='sharpe'))

## Equity curve và drawdown tại phí 0,25% mỗi chiều

In [ ]:
benchmark = pd.read_csv(INPUT / 'benchmark_vn30.csv', parse_dates=['date']).set_index('date')['return']
curves = {}
for strategy in ['MinVariance', 'MaxSharpe', 'EqualWeight']:
    periods = pd.read_csv(OUTPUT / f'{strategy}_fee_0.002500_periods.csv', parse_dates=['date']).set_index('date')
    curves[strategy] = periods['end_value']
dates = next(iter(curves.values())).index
curves['VN30'] = (1 + benchmark.reindex(dates)).cumprod()
equity = pd.DataFrame(curves)
drawdown = equity / equity.cummax() - 1

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
equity.plot(ax=axes[0], linewidth=1.8)
axes[0].set(title='Giá trị tích lũy — phí 0,25%', ylabel='Giá trị danh mục')
drawdown.plot(ax=axes[1], linewidth=1.4)
axes[1].set(title='Drawdown', xlabel='Tháng', ylabel='So với đỉnh')
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Benchmark chưa trừ phí và chưa được xác nhận độc lập là total return. Backtest cũng chưa mô hình hóa trượt giá, thuế, lô giao dịch, room ngoại, biên độ giá hoặc tạm ngừng giao dịch; vì vậy kết quả mang tính thăm dò, không phải khuyến nghị đầu tư.